# Neural Language Models — embeddings + softmax over the vocabulary

> Tutorial pair for [`neural_lm.py`](neural_lm.py). Contrast with the count-based
> [`ngram_lm.ipynb`](ngram_lm.ipynb).

## 1. Intuition
A count-based n-gram model treats every word as an unrelated symbol, so seeing
"the cat ran" tells it nothing about "the dog ran". A **neural** LM first maps
each word to a dense **embedding**, so similar words sit near each other and
share statistical strength. It then summarizes the history with a neural net and
predicts the next word with a softmax. Two flavors: a fixed-window MLP (Bengio
2003) and an LSTM that reads the whole history.

## 2. Concept (the slide)
- **Embed:** each word id $\to$ a learned vector $E[w]\in\mathbb R^{d}$.
- **Feed-forward LM:** concatenate the last $n-1$ embeddings, push through an MLP,
  softmax over the vocabulary. Context length is fixed.
- **Recurrent (LSTM) LM:** feed words one at a time; the hidden state carries an
  *unbounded* history. Predict the next token at every step.
- **Train** by minimizing cross-entropy (= negative log-likelihood of the true
  next word). **Perplexity** $=\exp(\text{cross-entropy})$.
- **Generate** by sampling from the softmax, sharpened/flattened by a
  *temperature*.

## 3. Math derivation

**Setup.** Same chain-rule factorization as any LM,
$P(w_1^T)=\prod_t P(w_t\mid w_1^{t-1})$, but now each conditional is produced by a
neural network with parameters $\theta$.

**Feed-forward LM.** With context $c=(w_{t-n+1},\dots,w_{t-1})$ and embedding
table $E$:
$$x=\big[E[w_{t-n+1}];\dots;E[w_{t-1}]\big],\quad
h=\tanh(W_1 x+b_1),\quad z=W_2 h+b_2.$$

**Softmax over the vocabulary.** The logits $z\in\mathbb R^{|V|}$ become a
distribution
$$P(w=i\mid c)=\operatorname{softmax}(z)_i=\frac{e^{z_i}}{\sum_{j} e^{z_j}}.$$

**Cross-entropy loss.** For the true next word $y$,
$$\mathcal L=-\log P(y\mid c)=-\log\operatorname{softmax}(z)_y= -z_y+\log\textstyle\sum_j e^{z_j}.$$
Its gradient w.r.t. the logits is the clean softmax-minus-one-hot:
$$\frac{\partial\mathcal L}{\partial z}=\operatorname{softmax}(z)-\mathbf 1_y,$$
which backpropagates through $W_2,h,W_1$ and finally **scatters** into the
embedding rows of the context words (each gets $\partial\mathcal L/\partial x$
for its slice). The module codes this by hand.

**Perplexity.** The standard intrinsic metric is the exponentiated average loss:
$$\mathrm{PP}=\exp\!\Big(\frac1N\sum_t\mathcal L_t\Big)
=\exp\!\Big(-\frac1N\sum_t\log P(w_t\mid w_1^{t-1})\Big).$$
So perplexity is *literally* $e$ raised to the cross-entropy — minimizing one
minimizes the other.

**Temperature sampling.** To generate, sample $w\sim\operatorname{softmax}(z/T)$.
$T\!\to\!0$ is greedy/argmax (safe, repetitive); $T=1$ is the model's own
distribution; $T>1$ flattens it (more surprising, more mistakes).

**Recurrent LM.** The LSTM replaces the fixed window: $h_t=\mathrm{LSTM}(E[w_t],
h_{t-1})$ and $z_t=W h_t+b$. The same softmax cross-entropy applies at every step;
because the recurrence is deep in time we **clip gradients** to stop them
exploding (see [`lstm.ipynb`](../../02.dl/rnn/lstm.ipynb)).

## 4. NumPy implementation — feed-forward LM with manual backprop

In [ ]:
# ===== actual implementation from neural_lm.py =====
from __future__ import annotations

import numpy as np

SEED = 0

def softmax(z):
    z = z - z.max(-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(-1, keepdims=True)

def _tokenize(text: str) -> list[str]:
    return text.lower().replace("\n", " ").split()

class FeedForwardLMNumPy:
    r"""
    Bengio-style neural n-gram LM. Predict w_t from the previous (n-1) words.

    Forward:
        x   = [E[w_{t-n+1}] ; ... ; E[w_{t-1}]]   (concatenated embeddings)
        h   = tanh(W_1 x + b_1)
        z   = W_2 h + b_2                          (logits over the vocabulary)
        p   = softmax(z)
    Loss = cross-entropy  -log p[w_t].

    The gradients are standard MLP backprop; the embedding gradient is scattered
    back to the rows of E for the words in the context window.
    """

    def __init__(self, n=3, dim=16, hidden=32, lr=0.3, seed=SEED):
        self.n, self.dim, self.hidden, self.lr, self.seed = n, dim, hidden, lr, seed

    def build_vocab(self, tokens):
        self.itos = sorted(set(tokens))
        self.stoi = {w: i for i, w in enumerate(self.itos)}
        self.V = len(self.itos)
        rng = np.random.default_rng(self.seed)
        self.E = rng.normal(0, 0.1, (self.V, self.dim))             # embeddings
        ctx = (self.n - 1) * self.dim
        self.W1 = rng.normal(0, 1 / np.sqrt(ctx), (ctx, self.hidden))
        self.b1 = np.zeros(self.hidden)
        self.W2 = rng.normal(0, 1 / np.sqrt(self.hidden), (self.hidden, self.V))
        self.b2 = np.zeros(self.V)
        return self

    def _windows(self, ids):
        for i in range(self.n - 1, len(ids)):
            yield ids[i - self.n + 1:i], ids[i]      # (context words, target)

    def fit(self, tokens, epochs=300):
        self.build_vocab(tokens)
        ids = [self.stoi[w] for w in tokens]
        data = list(self._windows(ids))
        rng = np.random.default_rng(self.seed)
        self.history = []
        for _ in range(epochs):
            rng.shuffle(data)
            loss = 0.0
            for ctx, y in data:
                x = self.E[ctx].reshape(-1)                  # concat embeddings
                a = x @ self.W1 + self.b1
                h = np.tanh(a)
                z = h @ self.W2 + self.b2
                p = softmax(z)
                loss += -np.log(p[y] + 1e-12)
                # ---- backprop ----
                dz = p.copy(); dz[y] -= 1.0                  # dL/dz (softmax-CE)
                dW2 = np.outer(h, dz); db2 = dz
                dh = self.W2 @ dz
                da = dh * (1 - h ** 2)                        # tanh'
                dW1 = np.outer(x, da); db1 = da
                dx = self.W1 @ da
                # ---- update ----
                self.W2 -= self.lr * dW2; self.b2 -= self.lr * db2
                self.W1 -= self.lr * dW1; self.b1 -= self.lr * db1
                dE = dx.reshape(self.n - 1, self.dim)
                for k, wid in enumerate(ctx):
                    self.E[wid] -= self.lr * dE[k]
            self.history.append(loss / len(data))
        return self

    def _logits(self, ctx_ids):
        x = self.E[ctx_ids].reshape(-1)
        h = np.tanh(x @ self.W1 + self.b1)
        return h @ self.W2 + self.b2

    def perplexity(self, tokens):
        ids = [self.stoi.get(w, 0) for w in tokens]
        total, n = 0.0, 0
        for ctx, y in self._windows(ids):
            p = softmax(self._logits(ctx))
            total += -np.log(p[y] + 1e-12); n += 1
        return float(np.exp(total / max(n, 1)))

    def generate(self, prefix, n_words=20, temperature=1.0, seed=SEED):
        rng = np.random.default_rng(seed)
        ids = [self.stoi[w] for w in prefix]
        for _ in range(n_words):
            ctx = ids[-(self.n - 1):]
            if len(ctx) < self.n - 1:                        # left-pad with the first id
                ctx = [ids[0]] * (self.n - 1 - len(ctx)) + ctx
            logits = self._logits(ctx) / max(temperature, 1e-6)
            p = softmax(logits)
            ids.append(int(rng.choice(self.V, p=p)))
        return [self.itos[i] for i in ids]

## 5. PyTorch implementation — an LSTM language model

In [ ]:
# ===== actual implementation from neural_lm.py =====
import torch

import torch.nn as nn

torch.set_num_threads(1)

def get_device():
    """CUDA > MPS > CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

class LSTMLanguageModel(nn.Module):
    """Embed -> LSTM -> linear softmax over the vocabulary (next-token prediction)."""

    def __init__(self, vocab, dim=32, hidden=64, layers=1):
        super().__init__()
        self.embed = nn.Embedding(vocab, dim)
        self.lstm = nn.LSTM(dim, hidden, num_layers=layers, batch_first=True)
        self.head = nn.Linear(hidden, vocab)
        self.vocab = vocab

    def forward(self, x, hidden=None):
        e = self.embed(x)                       # (B, T, dim)
        out, hidden = self.lstm(e, hidden)      # (B, T, hidden)
        return self.head(out), hidden           # logits (B, T, vocab)

    def fit(self, ids, seq_len=16, epochs=120, lr=0.005, batch=8):
        dev = get_device(); self.to(dev)
        opt = torch.optim.Adam(self.parameters(), lr=lr)
        loss_fn = nn.CrossEntropyLoss()
        ids = torch.as_tensor(ids, dtype=torch.long, device=dev)
        N = (len(ids) - 1) // seq_len
        X = ids[:N * seq_len].view(N, seq_len)
        Y = ids[1:N * seq_len + 1].view(N, seq_len)   # targets shifted by one
        self.history = []
        for _ in range(epochs):
            perm = torch.randperm(N, device=dev)
            total = 0.0
            for s in range(0, N, batch):
                idx = perm[s:s + batch]
                logits, _ = self(X[idx])
                loss = loss_fn(logits.reshape(-1, self.vocab), Y[idx].reshape(-1))
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(self.parameters(), 5.0)  # tame RNN grads
                opt.step()
                total += loss.item()
            self.history.append(total / max(1, N // batch))
        return self

    @torch.no_grad()
    def perplexity(self, ids):
        dev = next(self.parameters()).device
        x = torch.as_tensor(ids[:-1], dtype=torch.long, device=dev).unsqueeze(0)
        y = torch.as_tensor(ids[1:], dtype=torch.long, device=dev).unsqueeze(0)
        logits, _ = self(x)
        ce = nn.functional.cross_entropy(logits.reshape(-1, self.vocab), y.reshape(-1))
        return float(torch.exp(ce))

    @torch.no_grad()
    def generate(self, prefix_ids, n_words=20, temperature=1.0, seed=SEED):
        dev = next(self.parameters()).device
        g = torch.Generator(device="cpu").manual_seed(seed)
        ids = list(prefix_ids)
        hidden = None
        x = torch.as_tensor(ids, dtype=torch.long, device=dev).unsqueeze(0)
        for _ in range(n_words):
            logits, hidden = self(x, hidden)
            logits = logits[0, -1] / max(temperature, 1e-6)
            p = torch.softmax(logits, -1).cpu()
            nxt = int(torch.multinomial(p, 1, generator=g))
            ids.append(nxt)
            x = torch.as_tensor([[nxt]], dtype=torch.long, device=dev)
        return ids

def toy_text() -> str:
    base = (
        "the cat sat on the mat . "
        "the dog sat on the log . "
        "the cat chased the dog . "
        "the dog chased the cat . "
        "a happy cat naps on the mat . "
        "a happy dog runs in the park . "
        "the cat and the dog are friends . "
    )
    return base * 8

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    tokens = _tokenize(toy_text())
    cut = int(len(tokens) * 0.85)
    train_tok, test_tok = tokens[:cut], tokens[cut:]

    # ---- NumPy feed-forward LM ----
    ff = FeedForwardLMNumPy(n=3, dim=16, hidden=32, lr=0.3).fit(train_tok, epochs=150)
    print(f"NumPy FF-LM  train ppl = {ff.perplexity(train_tok):6.3f} | "
          f"test ppl = {ff.perplexity(test_tok):6.3f}")
    gen = ff.generate(["the", "cat"], n_words=12, temperature=0.7)
    print("  sample:", " ".join(gen))

    # ---- PyTorch LSTM LM (shares the NumPy vocab) ----
    ids = [ff.stoi[w] for w in train_tok]
    lm = LSTMLanguageModel(ff.V, dim=32, hidden=64).fit(ids, seq_len=12, epochs=120)
    test_ids = [ff.stoi.get(w, 0) for w in test_tok]
    print(f"Torch LSTM-LM             test ppl = {lm.perplexity(test_ids):6.3f}")
    prefix = [ff.stoi[w] for w in ["the", "cat"]]
    for temp in (0.5, 1.0):
        out = lm.generate(prefix, n_words=12, temperature=temp)
        print(f"  sample (T={temp}):", " ".join(ff.itos[i] for i in out))

## 6. Train / run — perplexity + sampled text from both models

In [ ]:
demo()

## 7. Visualization — training perplexity curve (cross-entropy -> ppl)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
import neural_lm as M

tok = M._tokenize(M.toy_text())
ff = M.FeedForwardLMNumPy(n=3, dim=16, hidden=32, lr=0.3).fit(tok, epochs=150)

plt.figure(figsize=(7, 4))
plt.plot(np.exp(ff.history))         # history stores mean cross-entropy/epoch
plt.xlabel("epoch"); plt.ylabel("training perplexity  exp(CE)")
plt.title("Neural n-gram LM: perplexity falls as the softmax sharpens")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- The neural LM's superpower is **generalization through embeddings** — unseen
  word combinations still get reasonable probability because similar words share
  geometry. No explicit smoothing needed.
- **Perplexity $=\exp(\text{cross-entropy})$**: report it, and compare only with
  matching vocabulary/tokenization.
- The **feed-forward** LM has a fixed context window (like an n-gram); the
  **LSTM** has unbounded context but needs gradient clipping and is sequential.
- On a tiny repeated toy text the LSTM can essentially **memorize** it (ppl near
  1) — on real data you must watch for overfitting (held-out perplexity, dropout,
  weight tying).
- Attention-based LMs ([Transformers](../../05.transformers/architectures/transformer.ipynb))
  drop recurrence for parallel, long-range context — today's state of the art.